# SciFact Graph Model Reranker

Ноутбук проверяет не отдельный поиск с нуля, а полный сценарий:

1. baseline `semantic search + graph rerank` формирует кандидатов;
2. графовая модель обучается переупорядочивать этих кандидатов;
3. GraphSAGE, GAT и R-GCN сравниваются с baseline по одним и тем же retrieval-метрикам;
4. выполняется небольшой автоматический подбор гиперпараметров.

Структура ноутбука следует методичке: разбиение выборки, преобразование данных в граф, описание архитектур, функция ошибки и регуляризация, обучение, промежуточные и финальные метрики, проверка переобучения, сравнение трех архитектур и подбор гиперпараметров.

Зависимости:

```python
%pip install pandas matplotlib numpy torch torch-geometric sentence-transformers
```

## Подготовка

In [ ]:
from __future__ import annotations

import copy
import json
import math
import random
import sys
import time
from collections import Counter, defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from sentence_transformers import SentenceTransformer
from torch import nn
from torch_geometric.data import Data
from torch_geometric.nn import GATConv, RGCNConv, SAGEConv

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_colwidth", 120)

NOTEBOOK_DIR = Path.cwd()
if NOTEBOOK_DIR.name != "notebooks":
    NOTEBOOK_DIR = Path.cwd() / "notebooks"

PROJECT_ROOT = NOTEBOOK_DIR.parent
PACKAGES_DIR = PROJECT_ROOT / "src" / "packages"
DATASET_DIR = NOTEBOOK_DIR / "data" / "scifact"
MODEL_CACHE_DIR = NOTEBOOK_DIR / "data" / "hf_cache"

if str(PACKAGES_DIR) not in sys.path:
    sys.path.insert(0, str(PACKAGES_DIR))

from sciguide_pipeline.infrastructure.processing import DeterministicEntityExtractor

## Конфигурация

In [ ]:
SEED = 42

MAX_DOCUMENTS = None
VALIDATION_SHARE = 0.2

CANDIDATES_PER_QUERY = 50
NEGATIVES_PER_POSITIVE = 6
MAX_ENTITIES_PER_TEXT = 8
MAX_ENTITY_NODES = 1500

EMBEDDING_MODEL_NAME = "BAAI/bge-m3"
GRAPH_SIGNAL_WEIGHT = 0.05
VECTOR_WEIGHT = 1.0 - GRAPH_SIGNAL_WEIGHT
ENTITY_MAX_DOCUMENT_FREQUENCY_SHARE = 0.08
NEIGHBOR_SIGNAL_WEIGHT = 0.15

DEFAULT_HIDDEN_DIM = 64
DEFAULT_DROPOUT = 0.25
DEFAULT_LR = 1e-3
WEIGHT_DECAY = 1e-4
EPOCHS = 25
PATIENCE = 6

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

## Данные и split

In [ ]:
def load_jsonl(path: Path) -> list[dict]:
    with path.open(encoding="utf-8") as file:
        return [json.loads(line) for line in file]


def load_qrels(path: Path) -> dict[str, dict[str, int]]:
    frame = pd.read_csv(path, sep="\t").rename(
        columns={"query-id": "query_id", "corpus-id": "corpus_id", "score": "relevance"}
    )
    qrels: dict[str, dict[str, int]] = defaultdict(dict)
    for row in frame.itertuples(index=False):
        qrels[str(row.query_id)][str(row.corpus_id)] = int(row.relevance)
    return dict(qrels)


def document_text(row: dict) -> str:
    return "\n\n".join(part.strip() for part in [row.get("title", ""), row.get("text", "")] if part and part.strip())


corpus_rows = load_jsonl(DATASET_DIR / "corpus.jsonl")
query_rows = load_jsonl(DATASET_DIR / "queries.jsonl")
corpus = {str(row["_id"]): row for row in corpus_rows}
queries = {str(row["_id"]): row["text"] for row in query_rows}
train_qrels = load_qrels(DATASET_DIR / "qrels" / "train.tsv")
test_qrels = load_qrels(DATASET_DIR / "qrels" / "test.tsv")

train_query_ids = list(train_qrels.keys())
random.shuffle(train_query_ids)
validation_size = int(len(train_query_ids) * VALIDATION_SHARE)
validation_query_ids = train_query_ids[:validation_size]
train_query_ids = train_query_ids[validation_size:]
test_query_ids = list(test_qrels.keys())

all_query_ids = train_query_ids + validation_query_ids + test_query_ids
all_qrels = {**train_qrels, **test_qrels}
relevant_doc_ids = {
    doc_id
    for query_id in all_query_ids
    for doc_id, rel in all_qrels.get(query_id, {}).items()
    if rel > 0
}

selected_doc_ids = list(corpus.keys()) if MAX_DOCUMENTS is None else list(sorted(relevant_doc_ids))
if MAX_DOCUMENTS is not None:
    for doc_id in corpus:
        if len(selected_doc_ids) >= MAX_DOCUMENTS:
            break
        if doc_id not in relevant_doc_ids:
            selected_doc_ids.append(doc_id)
    selected_doc_ids = selected_doc_ids[:MAX_DOCUMENTS]
selected_doc_id_set = set(selected_doc_ids)
doc_texts = {doc_id: document_text(corpus[doc_id]) for doc_id in selected_doc_ids}

pd.DataFrame(
    {
        "split": ["train", "validation", "test"],
        "queries": [len(train_query_ids), len(validation_query_ids), len(test_query_ids)],
        "documents_in_pool": [len(selected_doc_ids)] * 3,
    }
)

## Embeddings и графовый сигнал baseline

In [ ]:
embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME, cache_folder=str(MODEL_CACHE_DIR), device="cpu")

query_embeddings = embedding_model.encode([queries[qid] for qid in all_query_ids], normalize_embeddings=True, batch_size=8, show_progress_bar=True)
document_embeddings = embedding_model.encode([doc_texts[did] for did in selected_doc_ids], normalize_embeddings=True, batch_size=8, show_progress_bar=True)

query_embedding_by_id = dict(zip(all_query_ids, np.asarray(query_embeddings, dtype=np.float16)))
document_embedding_by_id = dict(zip(selected_doc_ids, np.asarray(document_embeddings, dtype=np.float16)))
document_matrix = np.asarray(document_embeddings, dtype=np.float32)
EMBEDDING_DIM = document_matrix.shape[1]
EMBEDDING_DIM

In [ ]:
extractor = DeterministicEntityExtractor()
query_entities_raw = {qid: tuple(extractor.extract(queries[qid])[:MAX_ENTITIES_PER_TEXT]) for qid in all_query_ids}
document_entities_raw = {did: tuple(extractor.extract(doc_texts[did])[:MAX_ENTITIES_PER_TEXT]) for did in selected_doc_ids}

entity_document_frequency: Counter[str] = Counter()
for entities in document_entities_raw.values():
    entity_document_frequency.update(set(entities))

max_entity_document_frequency = max(1, int(len(selected_doc_ids) * ENTITY_MAX_DOCUMENT_FREQUENCY_SHARE))
allowed_entities = {
    entity
    for entity, frequency in entity_document_frequency.items()
    if frequency <= max_entity_document_frequency
}
entity_idf = {
    entity: math.log(1 + len(selected_doc_ids) / (1 + entity_document_frequency[entity]))
    for entity in allowed_entities
}

query_entities = {
    qid: tuple(entity for entity in entities if entity in allowed_entities)
    for qid, entities in query_entities_raw.items()
}
document_entities = {
    did: tuple(entity for entity in entities if entity in allowed_entities)
    for did, entities in document_entities_raw.items()
}

entity_counts = Counter(entity for entities in list(query_entities.values()) + list(document_entities.values()) for entity in entities)
entity_ids = [entity for entity, _ in entity_counts.most_common(MAX_ENTITY_NODES)]
entity_id_set = set(entity_ids)

neighbors: dict[str, set[str]] = defaultdict(set)
for entities in document_entities.values():
    filtered = [entity for entity in entities if entity in entity_id_set]
    for left, right in zip(filtered, filtered[1:]):
        neighbors[left].add(right)
        neighbors[right].add(left)


def normalize(scores: np.ndarray) -> np.ndarray:
    if scores.max() == scores.min():
        return np.zeros_like(scores)
    return (scores - scores.min()) / (scores.max() - scores.min())


def graph_scores(query_id: str) -> np.ndarray:
    matched = {entity for entity in query_entities[query_id] if entity in entity_id_set}
    expanded = set(matched)
    for entity in matched:
        expanded.update(neighbors.get(entity, set()))

    scores = np.zeros(len(selected_doc_ids), dtype=np.float32)
    for i, doc_id in enumerate(selected_doc_ids):
        entities = set(document_entities[doc_id])
        exact_score = sum(entity_idf.get(entity, 0.0) for entity in entities & matched)
        neighbor_score = sum(entity_idf.get(entity, 0.0) for entity in entities & expanded if entity not in matched)
        scores[i] = exact_score + NEIGHBOR_SIGNAL_WEIGHT * neighbor_score
    return scores


def baseline_scores(query_id: str) -> np.ndarray:
    vector_scores = document_matrix @ query_embedding_by_id[query_id]
    structural_scores = graph_scores(query_id)
    return VECTOR_WEIGHT * normalize(vector_scores) + GRAPH_SIGNAL_WEIGHT * normalize(structural_scores)

pd.DataFrame(
    {
        "metric": ["graph_signal_weight", "vector_weight", "raw_entities", "filtered_entities", "entity_nodes", "neighbor_signal_weight"],
        "value": [GRAPH_SIGNAL_WEIGHT, VECTOR_WEIGHT, len(entity_document_frequency), len(allowed_entities), len(entity_ids), NEIGHBOR_SIGNAL_WEIGHT],
    }
)


## Кандидаты baseline и hard negatives

In [ ]:
baseline_rankings: dict[str, list[str]] = {}
for query_id in all_query_ids:
    ranked = [selected_doc_ids[i] for i in np.argsort(baseline_scores(query_id))[::-1]]
    positives = [doc_id for doc_id, rel in all_qrels.get(query_id, {}).items() if rel > 0 and doc_id in selected_doc_id_set]
    baseline_rankings[query_id] = list(dict.fromkeys(positives + ranked[:CANDIDATES_PER_QUERY]))


def dcg(relevances: list[float]) -> float:
    return sum(((2**rel) - 1) / math.log2(rank + 2) for rank, rel in enumerate(relevances))


def evaluate_rankings(rankings: dict[str, list[str]], query_ids: list[str], qrels: dict[str, dict[str, int]], k_values: tuple[int, ...] = (5, 10)) -> dict[str, float]:
    metrics = {f"ndcg@{k}": 0.0 for k in k_values} | {f"recall@{k}": 0.0 for k in k_values} | {f"mrr@{k}": 0.0 for k in k_values}
    evaluated = 0
    for query_id in query_ids:
        positives = {doc_id for doc_id, rel in qrels.get(query_id, {}).items() if rel > 0 and doc_id in rankings[query_id]}
        if not positives:
            continue
        for k in k_values:
            top_k = rankings[query_id][:k]
            hits = [doc_id in positives for doc_id in top_k]
            ideal = [1.0] * min(len(positives), k)
            metrics[f"ndcg@{k}"] += dcg([float(hit) for hit in hits]) / dcg(ideal)
            metrics[f"recall@{k}"] += sum(hits) / len(positives)
            metrics[f"mrr@{k}"] += next((1.0 / rank for rank, hit in enumerate(hits, start=1) if hit), 0.0)
        evaluated += 1
    return {name: round(value / evaluated, 4) for name, value in metrics.items()}


baseline_test_metrics = {"model": f"Semantic + graph baseline (w={GRAPH_SIGNAL_WEIGHT:.2f})", **evaluate_rankings(baseline_rankings, test_query_ids, test_qrels)}
pd.DataFrame([baseline_test_metrics])

## Граф для обучаемого reranker

In [ ]:
entity_embeddings = embedding_model.encode(entity_ids, normalize_embeddings=True, batch_size=16, show_progress_bar=True)
entity_embedding_by_id = dict(zip(entity_ids, np.asarray(entity_embeddings, dtype=np.float16)))

node_index: dict[tuple[str, str], int] = {}
node_features: list[np.ndarray] = []


def add_node(kind: str, external_id: str, vector: np.ndarray) -> int:
    key = (kind, external_id)
    if key not in node_index:
        node_index[key] = len(node_features)
        node_features.append(vector)
    return node_index[key]


for query_id in all_query_ids:
    add_node("query", query_id, query_embedding_by_id[query_id])
for doc_id in selected_doc_ids:
    add_node("document", doc_id, document_embedding_by_id[doc_id])
for entity_id in entity_ids:
    add_node("entity", entity_id, entity_embedding_by_id[entity_id])

relation_to_id = {
    "query_candidate": 0,
    "candidate_query": 1,
    "query_entity": 2,
    "entity_query": 3,
    "document_entity": 4,
    "entity_document": 5,
    "entity_cooccurs": 6,
    "entity_cooccurs_reverse": 7,
}
edges: list[tuple[int, int]] = []
edge_types: list[int] = []


def add_edge(source: int, target: int, relation: str) -> None:
    edges.append((source, target))
    edge_types.append(relation_to_id[relation])


for query_id, doc_ids in baseline_rankings.items():
    query_node = node_index[("query", query_id)]
    for doc_id in doc_ids:
        doc_node = node_index[("document", doc_id)]
        add_edge(query_node, doc_node, "query_candidate")
        add_edge(doc_node, query_node, "candidate_query")

for query_id, entities in query_entities.items():
    query_node = node_index[("query", query_id)]
    for entity in entities:
        if entity in entity_id_set:
            entity_node = node_index[("entity", entity)]
            add_edge(query_node, entity_node, "query_entity")
            add_edge(entity_node, query_node, "entity_query")

for doc_id, entities in document_entities.items():
    doc_node = node_index[("document", doc_id)]
    filtered = [entity for entity in entities if entity in entity_id_set]
    for entity in filtered:
        entity_node = node_index[("entity", entity)]
        add_edge(doc_node, entity_node, "document_entity")
        add_edge(entity_node, doc_node, "entity_document")
    for left, right in zip(filtered, filtered[1:]):
        add_edge(node_index[("entity", left)], node_index[("entity", right)], "entity_cooccurs")
        add_edge(node_index[("entity", right)], node_index[("entity", left)], "entity_cooccurs_reverse")

data = Data(
    x=torch.tensor(np.vstack(node_features), dtype=torch.float32),
    edge_index=torch.tensor(edges, dtype=torch.long).t().contiguous(),
    edge_type=torch.tensor(edge_types, dtype=torch.long),
).to(device)

pd.DataFrame({"metric": ["nodes", "edges", "relations"], "value": [data.num_nodes, data.num_edges, len(relation_to_id)]})

## Train pairs из baseline-кандидатов

In [ ]:
def make_pairs(query_ids: list[str], qrels: dict[str, dict[str, int]]) -> tuple[torch.Tensor, torch.Tensor]:
    pairs: list[tuple[int, int]] = []
    labels: list[float] = []
    for query_id in query_ids:
        candidates = baseline_rankings[query_id]
        positives = [doc_id for doc_id in candidates if qrels.get(query_id, {}).get(doc_id, 0) > 0]
        negatives = [doc_id for doc_id in candidates if doc_id not in set(positives)]
        if not positives or not negatives:
            continue
        query_node = node_index[("query", query_id)]
        for doc_id in positives:
            pairs.append((query_node, node_index[("document", doc_id)]))
            labels.append(1.0)
        for doc_id in negatives[: NEGATIVES_PER_POSITIVE * len(positives)]:
            pairs.append((query_node, node_index[("document", doc_id)]))
            labels.append(0.0)
    return torch.tensor(pairs, dtype=torch.long).t().contiguous().to(device), torch.tensor(labels, dtype=torch.float32).to(device)


train_pairs, train_labels = make_pairs(train_query_ids, train_qrels)
validation_pairs, validation_labels = make_pairs(validation_query_ids, train_qrels)

pd.DataFrame(
    {
        "split": ["train", "validation"],
        "pairs": [train_labels.numel(), validation_labels.numel()],
        "positive_share": [float(train_labels.mean().cpu()), float(validation_labels.mean().cpu())],
    }
)

## Модели

In [ ]:
class LinkDecoder(nn.Module):
    def __init__(self, hidden_dim: int, dropout: float):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(hidden_dim * 4, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 1),
        )

    def forward(self, z: torch.Tensor, pair_index: torch.Tensor) -> torch.Tensor:
        q = z[pair_index[0]]
        d = z[pair_index[1]]
        return self.mlp(torch.cat([q, d, torch.abs(q - d), q * d], dim=-1)).squeeze(-1)


class BaseReranker(nn.Module):
    def __init__(self, hidden_dim: int, dropout: float):
        super().__init__()
        self.decoder = LinkDecoder(hidden_dim, dropout)

    def forward(self, graph: Data, pair_index: torch.Tensor) -> torch.Tensor:
        return self.decoder(self.encode(graph), pair_index)


class GraphSAGEModel(BaseReranker):
    def __init__(self, input_dim: int, hidden_dim: int, dropout: float):
        super().__init__(hidden_dim, dropout)
        self.conv1 = SAGEConv(input_dim, hidden_dim)
        self.conv2 = SAGEConv(hidden_dim, hidden_dim)
        self.dropout = dropout

    def encode(self, graph: Data) -> torch.Tensor:
        x = F.relu(self.conv1(graph.x, graph.edge_index))
        return self.conv2(F.dropout(x, p=self.dropout, training=self.training), graph.edge_index)


class GATModel(BaseReranker):
    def __init__(self, input_dim: int, hidden_dim: int, dropout: float):
        super().__init__(hidden_dim, dropout)
        self.conv1 = GATConv(input_dim, hidden_dim, heads=2, concat=False, dropout=dropout)
        self.conv2 = GATConv(hidden_dim, hidden_dim, heads=1, concat=False, dropout=dropout)
        self.dropout = dropout

    def encode(self, graph: Data) -> torch.Tensor:
        x = F.elu(self.conv1(graph.x, graph.edge_index))
        return self.conv2(F.dropout(x, p=self.dropout, training=self.training), graph.edge_index)


class RGCNModel(BaseReranker):
    def __init__(self, input_dim: int, hidden_dim: int, dropout: float, num_relations: int):
        super().__init__(hidden_dim, dropout)
        self.conv1 = RGCNConv(input_dim, hidden_dim, num_relations=num_relations)
        self.conv2 = RGCNConv(hidden_dim, hidden_dim, num_relations=num_relations)
        self.dropout = dropout

    def encode(self, graph: Data) -> torch.Tensor:
        x = F.relu(self.conv1(graph.x, graph.edge_index, graph.edge_type))
        return self.conv2(F.dropout(x, p=self.dropout, training=self.training), graph.edge_index, graph.edge_type)

## Обучение и оценка reranking

In [ ]:
def model_rankings(model: nn.Module, query_ids: list[str]) -> dict[str, list[str]]:
    model.eval()
    rankings = {}
    with torch.no_grad():
        z = model.encode(data)
        for query_id in query_ids:
            doc_ids = baseline_rankings[query_id]
            pairs = torch.tensor(
                [(node_index[("query", query_id)], node_index[("document", doc_id)]) for doc_id in doc_ids],
                dtype=torch.long,
                device=device,
            ).t().contiguous()
            scores = model.decoder(z, pairs).cpu().numpy()
            rankings[query_id] = [doc_id for _, doc_id in sorted(zip(scores, doc_ids), reverse=True)]
    return rankings


def train_model(model: nn.Module, name: str, lr: float, weight_decay: float) -> tuple[nn.Module, pd.DataFrame]:
    model = model.to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    pos_weight = ((train_labels.numel() - train_labels.sum()) / train_labels.sum()).clamp(min=1.0)
    best_score = -1.0
    best_state = copy.deepcopy(model.state_dict())
    bad_epochs = 0
    history = []
    started_at = time.perf_counter()

    for epoch in range(1, EPOCHS + 1):
        model.train()
        optimizer.zero_grad()
        train_loss = F.binary_cross_entropy_with_logits(model(data, train_pairs), train_labels, pos_weight=pos_weight)
        train_loss.backward()
        optimizer.step()

        model.eval()
        with torch.no_grad():
            validation_loss = F.binary_cross_entropy_with_logits(model(data, validation_pairs), validation_labels, pos_weight=pos_weight)
        validation_metrics = evaluate_rankings(model_rankings(model, validation_query_ids), validation_query_ids, train_qrels)
        score = validation_metrics["ndcg@10"]

        if score > best_score:
            best_score = score
            best_state = copy.deepcopy(model.state_dict())
            bad_epochs = 0
        else:
            bad_epochs += 1

        history.append(
            {
                "model": name,
                "epoch": epoch,
                "train_loss": float(train_loss.cpu()),
                "validation_loss": float(validation_loss.cpu()),
                "loss_gap": float(validation_loss.cpu() - train_loss.cpu()),
                "elapsed_seconds": round(time.perf_counter() - started_at, 2),
                **validation_metrics,
            }
        )
        if bad_epochs >= PATIENCE:
            break

    model.load_state_dict(best_state)
    return model, pd.DataFrame(history)

## Сравнение трех архитектур

In [ ]:
def build_model(model_name: str, hidden_dim: int, dropout: float) -> nn.Module:
    if model_name == "GraphSAGE":
        return GraphSAGEModel(EMBEDDING_DIM, hidden_dim, dropout)
    if model_name == "GAT":
        return GATModel(EMBEDDING_DIM, hidden_dim, dropout)
    if model_name == "R-GCN":
        return RGCNModel(EMBEDDING_DIM, hidden_dim, dropout, num_relations=len(relation_to_id))
    raise ValueError(model_name)


histories = []
results = [baseline_test_metrics]
trained_models = {}

for model_name in ["GraphSAGE", "GAT", "R-GCN"]:
    model = build_model(model_name, DEFAULT_HIDDEN_DIM, DEFAULT_DROPOUT)
    trained_model, history = train_model(model, model_name, DEFAULT_LR, WEIGHT_DECAY)
    trained_models[model_name] = trained_model
    histories.append(history)
    rankings = model_rankings(trained_model, test_query_ids)
    results.append({"model": model_name, **evaluate_rankings(rankings, test_query_ids, test_qrels)})

history_frame = pd.concat(histories, ignore_index=True)
results_frame = pd.DataFrame(results)
results_frame

## Подбор гиперпараметров

По методичке подбираются минимум три гиперпараметра. Здесь автоматический grid search выполняется по `hidden_dim`, `dropout` и `lr` для лучшей архитектуры из предыдущего сравнения.

In [ ]:
best_model_name = results_frame.loc[results_frame["model"].str.startswith("Semantic + graph baseline").eq(False), "ndcg@10"].idxmax()
best_model_name = results_frame.loc[best_model_name, "model"]

search_grid = [
    {"hidden_dim": 64, "dropout": 0.15, "lr": 1e-3},
    {"hidden_dim": 96, "dropout": 0.25, "lr": 1e-3},
    {"hidden_dim": 128, "dropout": 0.35, "lr": 5e-4},
]

tuning_rows = []
for params in search_grid:
    model = build_model(best_model_name, params["hidden_dim"], params["dropout"])
    trained_model, history = train_model(model, f"{best_model_name} tuned", params["lr"], WEIGHT_DECAY)
    validation_score = history["ndcg@10"].max()
    test_metrics = evaluate_rankings(model_rankings(trained_model, test_query_ids), test_query_ids, test_qrels)
    tuning_rows.append({"model": best_model_name, **params, "best_validation_ndcg@10": validation_score, **test_metrics})

tuning_frame = pd.DataFrame(tuning_rows).sort_values("best_validation_ndcg@10", ascending=False)
tuning_frame

## Диагностика обучения

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

for model_name, frame in history_frame.groupby("model"):
    axes[0].plot(frame["epoch"], frame["train_loss"], label=model_name)
    axes[1].plot(frame["epoch"], frame["validation_loss"], label=model_name)
    axes[2].plot(frame["epoch"], frame["ndcg@10"], label=model_name)

axes[0].set_title("Train loss")
axes[1].set_title("Validation loss")
axes[2].set_title("Validation nDCG@10")
for axis in axes:
    axis.set_xlabel("Epoch")
    axis.legend()

plt.tight_layout()
plt.show()

## Проверка переобучения

In [ ]:
overfit_check = (
    history_frame.sort_values(["model", "ndcg@10"], ascending=[True, False])
    .groupby("model", as_index=False)
    .head(1)[["model", "epoch", "train_loss", "validation_loss", "loss_gap", "ndcg@10", "elapsed_seconds"]]
    .sort_values("ndcg@10", ascending=False)
)
overfit_check